# **Introduction to AI Workshop: Example 04**

In this example, we will construct a neural network that **generates text**. The goal of this neural network will be to predict the next word in a sequence of text.

For example, given the sequence of text: `I have`, the neural network might predict the next word to be `a`, `never`, or `always`.

We will train our neural network using a text file that we provide. The neural network will learn patterns in this text and use those patterns to generate new text.

Additionally, we will build our neural network based on the transformer architecture, which is also used in modern **large language models** (LLMs).




### **Part 1: Loading the Text Data**

First, we import the `TensorFlow` and `numpy` libraries. We also import the `re` library, which will help us work with text.

In [1]:
import tensorflow as tf
import numpy as np
import re

Next, we upload a text file that we will use to train our neural network.

In [2]:
from google.colab import files
uploaded = files.upload()

Saving TheRedHeadedLeague.txt to TheRedHeadedLeague.txt


We can now read the contents of the text file.

In [3]:
filename = next(iter(uploaded))
with open(filename, 'r', encoding='utf-8') as f:
    text = f.read()

Let's explore the text to get a better understanding of the data. **How many characters are in the text file? What does the text look like?**

In [4]:
print("Number of characters:", len(text))
print("\nFirst 500 characters:")
print(text[:500])

Number of characters: 49357

First 500 characters:
THE RED-HEADED LEAGUE

I had called upon my friend, Mr. Sherlock Holmes, one day in the
autumn of last year and found him in deep conversation with a
very stout, florid-faced, elderly gentleman with fiery red hair.
With an apology for my intrusion, I was about to withdraw when
Holmes pulled me abruptly into the room and closed the door
behind me.

"You could not possibly have come at a better time, my dear
Watson," he said cordially.

"I was afraid that you were engaged."

"So I am. Very much so


### **Part 2: Preparing the Text Data**

Before we can use the text to train a neural network, we need to convert it into a format that the neural network can understand.

First, we **convert all of the text to lowercase**. This ensures that words like `The` and `the` are treated as the same word.

We also **remove unnecessary whitespace**.

In [5]:
text = text.lower()
text = re.sub(r'\s+', ' ', text).strip()

Next, we divide the text into individual words and punctuation marks. This process is called **tokenization**.

Each individual word or punctuation mark is called a **token**.

In [6]:
tokens = re.findall(r"\w+|[^\w\s]", text)

Let's look at the first 50 tokens. Notice that we now have a list of individual tokens instead of one long string of text.

In [7]:
print("Number of tokens:", len(tokens))
print("\nFirst 50 tokens:")
print(tokens[:50])

Number of tokens: 11379

First 50 tokens:
['the', 'red', '-', 'headed', 'league', 'i', 'had', 'called', 'upon', 'my', 'friend', ',', 'mr', '.', 'sherlock', 'holmes', ',', 'one', 'day', 'in', 'the', 'autumn', 'of', 'last', 'year', 'and', 'found', 'him', 'in', 'deep', 'conversation', 'with', 'a', 'very', 'stout', ',', 'florid', '-', 'faced', ',', 'elderly', 'gentleman', 'with', 'fiery', 'red', 'hair', '.', 'with', 'an', 'apology']


### **Part 3: Creating a Vocabulary**

A neural network cannot work directly with words. Therefore, we need to assign a unique integer to each token.

First, we create a **vocabulary** containing all of the unique tokens in our text.

In [8]:
vocabulary = sorted(set(tokens))
vocab_size = len(vocabulary)
print("Vocabulary size:", vocab_size)

Vocabulary size: 1930


We also include a special token called `<UNK>`, which we will use for words that are not in our vocabulary.

In [9]:
vocabulary = ["<UNK>"] + vocabulary
vocab_size = len(vocabulary)
print("Vocabulary size:", vocab_size)

Vocabulary size: 1931


Let's look at the first 30 entries in our vocabulary.

In [10]:
print(vocabulary[:30])

['<UNK>', '!', '"', "'", ',', '-', '.', '000', '17', '1890', '2', '27', '30', '4', '7', '9', ':', ';', '?', 'a', 'abbots', 'able', 'about', 'above', 'abruptly', 'absence', 'absolute', 'absorbing', 'abutted', 'accomplice']


We can now assign an integer to each token.

We will use two dictionaries to do this. The first dictionary converts words into integers, and the second dictionary converts integers back into words.



In [11]:
word_to_int = {
    word: i for i, word in enumerate(vocabulary)
}

In [12]:
int_to_word = {
    i: word for i, word in enumerate(vocabulary)
}

We can now convert our entire text into a sequence of integers.

In [13]:
encoded_text = np.array([
    word_to_int[word] for word in tokens
])

Let's look at the first 30 tokens and their corresponding numerical representations. **What is the numerical representation of token `the`? What token corresponds to integer `4`?**

In [14]:
print("First 30 tokens:")
print(tokens[:30])
print("\nCorresponding numerical representation:")
print(encoded_text[:30])

First 30 tokens:
['the', 'red', '-', 'headed', 'league', 'i', 'had', 'called', 'upon', 'my', 'friend', ',', 'mr', '.', 'sherlock', 'holmes', ',', 'one', 'day', 'in', 'the', 'autumn', 'of', 'last', 'year', 'and', 'found', 'him', 'in', 'deep']

Corresponding numerical representation:
[1701 1395    5  816  984  869  785  248 1787 1128  713    4 1120    6
 1511  845    4 1197  434  885 1701  132 1184  972 1917   78  700  834
  885  442]


We can also convert these integers back into words to make sure that our encoding worked correctly.

In [15]:
decoded = [
    int_to_word[i] for i in encoded_text[:30]
]
print(decoded)

['the', 'red', '-', 'headed', 'league', 'i', 'had', 'called', 'upon', 'my', 'friend', ',', 'mr', '.', 'sherlock', 'holmes', ',', 'one', 'day', 'in', 'the', 'autumn', 'of', 'last', 'year', 'and', 'found', 'him', 'in', 'deep']


### **Part 4: Preparing the Training Data**

Now that we have converted our text into a sequence of integers, we need to prepare the data that we will use to train our neural network.

Remember that our goal is to predict the next word in a sequence of text.

For example, suppose we have the following text: `I have a dog that likes to play`. We want our neural network to learn that, given the words `I have a dog`, the next word is `that`.

To accomplish this, we divide our text into smaller sequences. We define the length of each sequence as the **block size**.

In [16]:
BLOCK_SIZE = 20

Next, we construct our **training data**.

For each sequence of 21 tokens, we use the first 20 tokens as the input (`X`) and the last 20 tokens as the expected output (`y`).

In [17]:
sequences = np.array([
    encoded_text[i:i + BLOCK_SIZE + 1]
    for i in range(len(encoded_text) - BLOCK_SIZE)
])
X = sequences[:, :-1]
y = sequences[:, 1:]

Let's explore our training data. **How many input sequences do we have? How many tokens are in each input sequence?**

In [18]:
print("Input shape:", X.shape)
print("Output shape:", y.shape)

Input shape: (11359, 20)
Output shape: (11359, 20)


We can also look at the first input sequence and its corresponding expected output.

Notice that the expected output is the same sequence of words as the input, **shifted by one token**. This is how we teach our neural network to predict the next token in a sequence.

In [19]:
print("Input:")
print([int_to_word[i] for i in X[0]])
print("\nExpected output:")
print([int_to_word[i] for i in y[0]])

Input:
['the', 'red', '-', 'headed', 'league', 'i', 'had', 'called', 'upon', 'my', 'friend', ',', 'mr', '.', 'sherlock', 'holmes', ',', 'one', 'day', 'in']

Expected output:
['red', '-', 'headed', 'league', 'i', 'had', 'called', 'upon', 'my', 'friend', ',', 'mr', '.', 'sherlock', 'holmes', ',', 'one', 'day', 'in', 'the']


### **Part 5: Designing the Neural Network**

We are now ready to design our neural network. Unlike the neural networks we constructed in the previous examples, this neural network will use a **transformer architecture**.

The **transformer architecture** uses a mechanism called **attention** to determine which tokens in a sequence are important when predicting the next token.

We will use three main components:

* An **embedding layer**, which converts each token into a vector representation (i.e., a list of numbers).

* An **attention layer**, which allows the neural network to consider the relationships between tokens.

* An **output layer**, which predicts the next token.

First, we define a few parameters for our neural network.

The **embedding** dimension determines the size of the vector representation of each token.

The number of **attention** heads determines how many different attention calculations the neural network performs.

The **feed-forward** dimension determines the number of neurons in an additional layer of the network.

In [20]:
EMBEDDING_DIM = 64
NUM_HEADS = 4
FF_DIM = 128

Next, we define the input layer. Each input will contain 20 integers, representing 20 tokens.

In [21]:
inputs = tf.keras.Input(shape=(BLOCK_SIZE,))

Next, we add an **embedding** layer.

The **embedding** layer converts each token into a vector of numbers. These vectors are learned during training and allow the neural network to represent words in a way that captures relationships between them.

In [22]:
x = tf.keras.layers.Embedding(
    vocab_size,
    EMBEDDING_DIM
)(inputs)

We also need to give the neural network information about the position of each word in the sequence. This is important because the meaning of a sentence can depend on the order of the words.

We then add **positional embeddings** to the word embeddings.

In [23]:
positions = tf.range(BLOCK_SIZE)
position_embeddings = tf.keras.layers.Embedding(
    BLOCK_SIZE,
    EMBEDDING_DIM
)(positions)
x = x + position_embeddings

Next, we add an **attention** layer.

**Attention** allows the neural network to consider the relationships between words when making predictions.

We use a **causal attention mask**, which prevents the neural network from looking at future words when predicting the next word. This is important because, when generating text, the neural network will only have access to the words that have already been generated.

In [24]:
attention = tf.keras.layers.MultiHeadAttention(
    num_heads=NUM_HEADS,
    key_dim=EMBEDDING_DIM // NUM_HEADS
)(x, x, use_causal_mask=True)

We add the **attention** output to the original input and normalize the result.

This helps the neural network retain information from the previous layer while learning new relationships between words.

In [25]:
x = tf.keras.layers.Add()([x, attention])
x = tf.keras.layers.LayerNormalization()(x)

Next, we add a **feed-forward** layer. This layer processes the information produced by the attention layer.

In [26]:
ff = tf.keras.layers.Dense(
    FF_DIM,
    activation='relu'
)(x)
ff = tf.keras.layers.Dense(
    EMBEDDING_DIM
)(ff)

We again add the output to the previous layer and normalize the result.

In [27]:
x = tf.keras.layers.Add()([x, ff])
x = tf.keras.layers.LayerNormalization()(x)

Finally, we add the output layer.

The number of neurons in this layer should match the number of unique tokens in our vocabulary. Each neuron represents a possible next word.

In [28]:
outputs = tf.keras.layers.Dense(vocab_size)(x)

We can now create our neural network and examine its architecture.

In [29]:
model = tf.keras.Model(inputs, outputs)
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 20)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding           │ (None, 20, 64)    │    123,584 │ input_layer[0][0] │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 20, 64)    │          0 │ embedding[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 20, 64)    │     16,640 │ add[0][0],        │
│ (MultiHeadAttentio… │                   │            │ add[0][0]         │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_1 (Add)         │ (None, 20, 64)    │          0 │ add[0][0],        │
│                     │                   │            │ multi_head_atten… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalization │ (None, 20, 64)    │        128 │ add_1[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 20, 128)   │      8,320 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 20, 64)    │      8,256 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_2 (Add)         │ (None, 20, 64)    │          0 │ layer_normalizat… │
│                     │                   │            │ dense_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 20, 64)    │        128 │ add_2[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 20, 1931)  │    125,515 │ layer_normalizat… │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 282,571 (1.08 MB)

 Trainable params: 282,571 (1.08 MB)

 Non-trainable params: 0 (0.00 B)

### **Part 6: Training the Neural Network**

Before training our neural network, we need to compile it by specifying the loss function and optimization algorithm.

We will use the Adam optimization algorithm and sparse categorical crossentropy as our loss function.

In [30]:
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [31]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.001
    ),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(
        from_logits=True
    ),
    metrics=['accuracy']
)

We then train our neural network using the training data.

In [32]:
history = model.fit(
    X, y,
    epochs=20,
    batch_size=32
)

Epoch 1/20
355/355 ━━━━━━━━━━━━━━━━━━━━ 18s 21ms/step - accuracy: 0.2349 - loss: 4.4104
Epoch 2/20
355/355 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.3728 - loss: 2.7482
Epoch 3/20
355/355 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.5338 - loss: 1.9097
Epoch 4/20
355/355 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.6963 - loss: 1.2471
Epoch 5/20
355/355 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.7872 - loss: 0.8876
Epoch 6/20
355/355 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.8326 - loss: 0.7049
Epoch 7/20
355/355 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.8587 - loss: 0.5988
Epoch 8/20
355/355 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.8744 - loss: 0.5296
Epoch 9/20
355/355 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.8850 - loss: 0.4862
Epoch 10/20
355/355 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.8913 - loss: 0.4533
Epoch 11/20
355/355 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.8982 - loss: 0.4262
Epoch 12/20
355/355 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/st

Notice that the loss decreases as the neural network learns to predict the next word. The **accuracy** tells us how often the neural network correctly predicts the next word.

### **Part 7. Generating Text**

Now that our neural network has been trained, we can use it to **generate new text**.

We begin by providing the neural network with a starting sequence of words, which we call a **prompt**.

The neural network uses this prompt to predict the next word. We then add that word to the sequence and ask the neural network to predict another word.

We repeat this process to generate a longer sequence of text.



First, we define a function to **generate text**.

In [33]:
def generate_text(prompt, length=50, temperature=0.8):

    # Convert the prompt into tokens
    prompt_tokens = re.findall(
        r"\w+|[^\w\s]",
        prompt.lower()
    )

    # Convert tokens into integers
    generated = [
        word_to_int.get(word, word_to_int["<UNK>"])
        for word in prompt_tokens
    ]

    # Generate one word at a time
    for _ in range(length):

        # Keep the most recent 20 words
        context = generated[-BLOCK_SIZE:]

        # Add padding if necessary
        context = [0] * (BLOCK_SIZE - len(context)) + context

        # Predict the next word
        prediction = model.predict(
            np.array([context]),
            verbose=0
        )

        # Get the prediction for the last position
        logits = prediction[0, -1, :]

        # Prevent the model from predicting the padding token
        logits[0] = -1e9

        # Adjust the predictions using temperature
        logits = logits / temperature

        # Randomly select the next word
        next_word = tf.random.categorical(
            [logits], num_samples=1
        )[0, 0].numpy()

        generated.append(int(next_word))

    # Convert integers back into words
    generated_words = [
        int_to_word[i] for i in generated
    ]

    # Combine the words into a string
    result = " ".join(generated_words)

    # Remove spaces before punctuation
    result = re.sub(r'\s+([.,!?;:])', r'\1', result)

    return result

Let's try generating some text.

We start with the prompt `I have never` and ask our neural network to generate 100 additional words.

In [34]:
prompt = "I have never"
generated_text = generate_text(prompt, length=100)
print(generated_text)

i have never heard of any such body. then i asked him who mr. duncan ross was. he answered that the name was new to him. " ' well, ' said i, ' the gentleman at no. 4. ' " ' what, the red - headed man? ' " ' why that? ' i asks. " ' says he, ' here ' s another vacancy on the league of the red - headed men. it ' s worth quite a little fortune to any man who gets it,


**Try a different prompt!** Notice that the generated text is influenced by the text that we used to train our neural network. However, remember that our neural network is relatively small and has only been trained on one text file, so the generated text may not make much sense.

### **Part 8. Experimenting with Temperature**
One interesting parameter we can change when generating text is called **temperature**.

**Temperature** controls how much randomness there is when the neural network selects the next word.


* A **low temperature** makes the neural network more likely to select words with high predicted probabilities.

* A **high temperature** makes the neural network more likely to select words with lower predicted probabilities.

Let's see how temperature affects the text generated by our neural network.

In [35]:
prompt = "I have never"
for temperature in [0.2, 0.8, 1.5, 2.0]:
    print("Temperature:", temperature)
    generated_text = generate_text(prompt, length=100, temperature=temperature)
    print(generated_text + "\n")

Temperature: 0.2
i have never heard of any such body. then i asked him who mr. duncan ross was. he answered that the name was new to him. " ' well, ' said i, ' the gentleman at no. 4. ' " ' what, the red - headed man? ' " ' yes. ' oh, ' said he, ' his name was william morris. he was a solicitor and was using my room as a temporary convenience until his new premises were ready. he moved out yesterday. ' "

Temperature: 0.8
i have never heard of any 30 pounds. " " the advertisement. " oh, he has his faults, too, " said mr. wilson. " never was such a fellow for photography. snapping away to open and a camera when he ought to be improving his mind, and then diving down into the cellar like a rabbit into its hole to develop his pictures. that is his main arteries would have him so pleased at him fortune. " well, when i thought over the matter all day, and by rubber after

Temperature: 1.5
i have never dared to leave the room for an instant, for i should not apply. ' ll see for months on end

Notice how the generated text changes as we increase the temperature. **What happens when the temperature is very low? What happens when the temperature is very high?**

### **Conclusion**

In this example, we constructed and trained a small language model from scratch.

We started with a text file, converted the text into numerical representations, and used these representations to train a neural network to predict the next word in a sequence. We then used the trained neural network to generate new text.

Although our neural network is relatively simple, it uses the same basic idea of predicting the next word that is used by modern LLMs.

In the next example, we will explore how to use a **pre-trained LLM** to generate text. Unlike the neural network we constructed here, a **pre-trained LLM** has already been trained on a very large amount of text and can generate much more sophisticated responses.